# Machine Learning Example

This is a markdown, or text cell.  It's for humans, not for the computer.

* we can add bulleted lists
* that include details
* we can add **bold**, *italic*, and `monospace` 

This is helpful for documentation and remembering what the heck you did and why in readable, well-formatted and attractive human language.

In [0]:
# This is a code cell!  It's for the computer.  We can "run" it by pressing the "Run" button above

# Step 1: Import extra libraries we need.  (this is a comment)
# we give them a shorter, easier name so we can use them later
import numpy as np
import pandas as pd

#  we can also pick and choose the little bits of things we want:

from sklearn.datasets import load_breast_cancer        # this is sample data
from sklearn.model_selection import train_test_split   # this is how we split data into train/test.
from sklearn.linear_model import LogisticRegression    # this is our algorithm!
from sklearn.metrics import accuracy_score


## Breast Cancer Classification

We're going to work with some [sample breast cancer data](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) that has a lot of information about images of tissue cells (30 features).  We also know if the tissue sample went on to be malignant or benign (the label).

We'll pull that data into some objects that we can work with and look at.

In [0]:
# Step 2: Load the dataset
data = load_breast_cancer()

X = data.data      # features (measurements)
y = data.target    # labels (0 = malignant, 1 = benign)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


It might be interesting to understand what our features **are** and to get a quick peek at them.  

I have a hard time remembering all the Python I used to write, so I make ample use of the Databricks "generate" to use AI to help me write Python.  For example, a bit below here I generated some code by asking Databricks to help me generate code for "show the first five values for the features and label them with the feature names."

If we were really doing a true research project, we'd want to do what's called feature reduction -- because a lot of these features are kind of tangled up with each other and measure similar things (in math terms, they are colinear, or correlated).  But for now we're going to skip all the nuances.

In [0]:
# Optional: See the feature names
print(data.feature_names)


In [0]:
pd.DataFrame(X[:5], columns=data.feature_names)

And let's just check our labels (the cancer outcome).

In [0]:
# Optional: See the target names
print(data.target_names)

In [0]:
# Let's look at the first few labels.  They're not stored with the full label,
# so let's see how they're stored (hint: it's a number) and their labels.

labels_df = pd.DataFrame({
    "raw_value": y[:5],
    "name": [data.target_names[val] for val in y[:5]]
})
display(labels_df)

## Prepare for learning

We need to split our data, what we already know, into training data (what we'll use to build a model) and test data (what we'll use to see if what we built is any good).

We'll do a test size of 20%, or 0.2.

In [0]:
# Step 3: Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## Picking an algorithm

Picking an algorithm is a bit of an art -- it helps if you have a stats background.  In our case, we're going to use a logistic (2 outcomes) regression model.  This will fit an S-shaped curve that will help us divide our cases into two distinct values -- benign or malignant.

We're going to help our program here know when to quit.  Once it's stopped improving, it can quit, or it should quit when it's tried 5000 times to get the right values.

In [0]:
# Step 4: Create the model
model = LogisticRegression(max_iter=5000)

# Step 5: Train the model
model.fit(X_train, y_train)


OK, so we have a model.  Let's drop in our test cases and see how it performs on data it hasn't seen already.

In [0]:
# Step 6: Make predictions
y_pred = model.predict(X_test)

print("First 10 predictions:", y_pred[:10])
print("First 10 actual values:", y_test[:10])


That looks pretty good!  Now, when we think about accuracy metrics, we have to think about what kinds of errors really matter.  Do we want to optimize for fewest errors overall?  Fewest false positives?  Fewest false negatives?  It really depends on the case.

There are lots of ways to evaluate model performance and we won't get into that here.  Unless you tell your program otherwise, it's going to assume you want overall accuracy (reducing all errors equally).

Let's look at accuracy.

In [0]:
# Step 7: Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print("Model accuracy:", accuracy)


We can look at true labels versus predictions using a **confusion matrix**.  Here, I've made a pretty one that has labels to help you understand it.

In [0]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Generate confusion matrix values
cm = confusion_matrix(y_test, y_pred)

# Create display with clear labels
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malignant", "Benign"]
)

# Plot
fig, ax = plt.subplots()
disp.plot(ax=ax)

plt.title("Confusion Matrix: Breast Cancer Classifier")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

# Add text annotations for confusion matrix cells
labels = [
    "True Positive", "False Negative",
    "False Positive", "True Negative"
]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"\n\n{labels[i*2+j]}", 
                ha="center", va="top", color="#CCC", fontsize=10, fontweight="bold")

plt.show()

That's a high accuracy rate, with good TP and TN rates.

But what the heck *is* this model?

What's inside this "black box"?

In [0]:
# Step 8: Examine model coefficients

coefficients = pd.DataFrame({
    "Feature": data.feature_names,
    "Coefficient": model.coef_[0]
})

# Sort by absolute value (most important features first)
coefficients["Abs_Coefficient"] = np.abs(coefficients["Coefficient"])
coefficients = coefficients.sort_values("Abs_Coefficient", ascending=False)

coefficients.head(10)


We have a model that is explainable (we can show the model coefficients), and that works well.  I think we can call this success -- we could probably use this as an "auto-sorter" to pre-label likely cancer cells for pathology.